# WISE² RECAST ENGINE × OpenJutsu — Free Google Colab GPU Backend

**Purpose:** Run ComfyUI + OpenJutsu custom nodes on a free Google Colab GPU (T4 15 GB VRAM / L4 24 GB VRAM) and expose a temporary HTTPS Cloudflare Quick Tunnel URL to your ThinkPad X1 or iPhone WISE² Command Center.

**Resource Reality Check:**
- Free Colab T4 has **15 GB VRAM** and **~12.7 GB system RAM** (~3-4 hr session limit).
- Ideal for **480p 4-6s clips**, **FLUX.3 Klein relighting**, **Fix Colours**, and **SAM 3.1 mask checks**.
- Full MiniMax H3 INT8 streaming benefits from 30-64 GB RAM (see Kaggle T4x2 30 GB RAM notebook or self-hosted GPU for long 736p/1080p renders).

In [ ]:
# 1. Verify GPU & System Resources
!nvidia-smi
!free -h
!df -h /content

In [ ]:
# 2. Install ComfyUI (shallow clone) + Audited OpenJutsu Custom Nodes
import os, subprocess
%cd /content
if not os.path.exists('/content/ComfyUI'):
    !git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

CUSTOM_NODES = [
    'https://github.com/D3voz/comfyui-h3-oneclick-swap.git',
    'https://github.com/D3voz/comfyui-websocket-reconnect-fix.git',
    'https://github.com/D3voz/comfyui-image-detail-contrast.git',
    'https://github.com/ethanfel/ComfyUI-MiniMaxH3-Context-Loop.git',
    'https://github.com/Adudeguyman/ComfyUI-Fantastic-MiniMaxH3-PromptBuilder.git',
    'https://github.com/kijai/ComfyUI-KJNodes.git',
    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'https://github.com/orion4d/Orion4D_FXMax.git',
    'https://github.com/Fannovel16/comfyui_controlnet_aux.git',
]
os.makedirs('/content/ComfyUI/custom_nodes', exist_ok=True)
for repo in CUSTOM_NODES:
    name = repo.rstrip('/').split('/')[-1].replace('.git', '')
    target = f'/content/ComfyUI/custom_nodes/{name}'
    if not os.path.exists(target):
        subprocess.run(['git', 'clone', '--depth', '1', repo, target], check=True)
    req = f'{target}/requirements.txt'
    if os.path.exists(req):
        subprocess.run(['pip', 'install', '-q', '-r', req], check=False)
print('Custom nodes staged cleanly.')

In [ ]:
# 3. Launch ComfyUI on port 8188 + Cloudflare Quick Tunnel (FREE, no account required)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, time, re
comfy = subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--lowvram'], cwd='/content/ComfyUI')
time.sleep(8)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n====================================================')
        print('PASTE THIS URL INTO WISE2 RECAST -> GPU BACKENDS -> GOOGLE COLAB:')
        print(m.group(0))
        print('====================================================\n')
        break
comfy.wait()